# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² mass spectrometry protein dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. We'll examine the dataset structure, extract records, process numeric fields, and visualize the data.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure required library is installed (run once)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# For visualization
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load croissant dataset (metadata and manifest)
dataset = mlc.Dataset(croissant_url)

# Retrieve the metadata
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review available record sets and their field and column `@id`s. All subsequent operations will refer to entities by their Croissant `@id` according to best practice.

Let's list the available record sets, and for each, show the included fields and columns.

In [ ]:
# List all record sets by their @id
recordset_ids = [rs['@id'] for rs in metadata.record_sets]
print('Available record sets:')
for rs in metadata.record_sets:
    print(f"  RecordSet  @id: {rs['@id']} | Name: {rs.get('name', '(no name)')}")

# For each record set, print its fields and columns, referenced by @id
for rs in metadata.record_sets:
    print(f"\nRecordSet @id: {rs['@id']}  ({rs.get('name','')})")
    # List fields
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for f in fields:
            # Pull the field object from metadata.fields by @id
            field_id = f['@id'] if isinstance(f, dict) else f
            # Try to find the field object definition
            field_obj = None
            for fo in metadata.fields:
                if fo['@id'] == field_id:
                    field_obj = fo
                    break
            fname = field_obj.get('name', '') if field_obj else ''
            col_ids = [col['@id'] if isinstance(col, dict) else col for col in (field_obj.get('column', []) if field_obj else [])]
            print(f"    Field @id: {field_id} | name: {fname} | columns: {col_ids}")
    else:
        print("  (No fields)")

## 3. Data Extraction
We extract records from each record set and load them into pandas DataFrames using their Croissant `@id` for reference. All entity references are via `@id`.

Below, you can select which record set to examine in detail.

In [ ]:
# List of available record set @id
record_sets = recordset_ids  # e.g. ['@id1', '@id2', ...]

# Extract data for each record set
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    else:
        print("  No records found.")
print('\nDataFrames available for record sets:', list(dataframes.keys()))

Now let's pick the primary data record set (with peptides/proteins) to examine—**replace this `@id` for your analysis** if needed.

We'll peek at the first few rows and column names.

In [ ]:
# Select the main table: use its @id (e.g., protein data RecordSet)
# For this dataset, the main RecordSet @id is likely the only or main table; we'll use the first.
main_record_set_id = record_sets[0]
df = dataframes[main_record_set_id]

print(f"Columns for record set {main_record_set_id}:")
print(df.columns.tolist())

df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data filtering, normalization, and grouping using fields referenced by their Croissant `@id`. Adjust field `@id`s for the ones relevant to your target analysis, referring to the Data Overview.

In [ ]:
# Identify a numeric field and a grouping field
# Example: Let's say 'Coverage_percent' is column with @id='Coverage_percent', and 'Accession' is an identifier field

numeric_field_id = None
group_field_id = None
# Attempt to suggest a numeric field automatically
for col in df.columns:
    if 'percent' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower() or 'peptide' in col.lower():
        try:
            # Test if column is numeric
            pd.to_numeric(df[col].dropna().head(10))
            numeric_field_id = col
            break
        except:
            continue
# Suggest group field
for col in df.columns:
    if 'accession' in col.lower() or 'id' in col.lower():
        group_field_id = col
        break
if not numeric_field_id:
    numeric_field_id = df.select_dtypes('number').columns[0]
if not group_field_id:
    group_field_id = df.columns[0]

print(f"Numeric field suggested: {numeric_field_id}")
print(f"Group field suggested: {group_field_id}")

# Remove missing/NaN values
df_eda = df.copy()
df_eda = df_eda[pd.to_numeric(df_eda[numeric_field_id], errors='coerce').notnull()]
df_eda[numeric_field_id] = pd.to_numeric(df_eda[numeric_field_id])

# Filter for values above a threshold (e.g., >10)
threshold = 10
filtered_df = df_eda[df_eda[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (count={len(filtered_df)}):")
print(filtered_df[[group_field_id, numeric_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field and compute group means
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id}, showing mean {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. We'll plot the distribution of the selected numeric field for filtered data and a scatter plot if possible.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id} (> {threshold})')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If the DataFrame includes a second numeric column, plot against it
other_numeric_cols = filtered_df.select_dtypes('number').columns.drop([numeric_field_id, f"{numeric_field_id}_normalized"], errors='ignore')
if len(other_numeric_cols) > 0:
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=filtered_df[numeric_field_id], y=filtered_df[other_numeric_cols[0]])
    plt.xlabel(numeric_field_id)
    plt.ylabel(other_numeric_cols[0])
    plt.title(f'Scatter: {numeric_field_id} vs {other_numeric_cols[0]})')
    plt.show()

## 6. Conclusion

- We loaded and explored the protein dataset defined by a Croissant schema, referencing all entities by their `@id`.
- We identified key record sets, fields, and numeric columns, applied filtering, normalization, and basic grouping.
- Visualization highlighted the distribution of a central numeric field and the relationship between main attributes.

For more advanced analysis, consult the field definitions and dataset documentation for domain interpretation!